# Part 2-4. 센서 데이터 로그 분석 및 가공

드론 한 대가 5분 동안 비행하면서 남긴 **로그 두 개**를 분석한다.
같은 비행을 두 시스템이 각자 기록했다.

| 파일 | 어디서 나온 것 | 시각 기준 | 좌표계 |
|---|---|---|---|
| `flight.mcap` | ROS2 (rosbag2) | 1970년 기준 나노초 | ENU (동-북-상) |
| `flight.ulg` | PX4 비행제어 소프트웨어 | 부팅 후 마이크로초 | NED (북-동-하) |

**진행 방식** — 과제 1부터 7까지 순서대로 푼다. 각 과제의 코드 셀에서 `# IMPLEMENT HERE` 자리를 채운다.
바로 앞 과제의 결과를 다음 과제가 그대로 쓴다. 셀은 위에서 아래로 한 번씩 실행한다.

**쓰는 도구** — Part 1(모듈과 패키지), Part 2-1(Matplotlib), 2-2(NumPy), 2-3(Pandas)에서 배운 것만 쓴다.
로그 파일을 여는 코드는 `loaders.py` 로 주어진다. 직접 만들 필요 없다.

**제출물** — 채운 이 노트북, `out/clean.csv`, `out/segments.csv`, `out/dashboard.png`, `flightkit/` 폴더.

In [ ]:
# 실습 준비. 이 셀만 먼저 실행한다 (Colab 기준 30초 정도)
!pip -q install mcap mcap-ros2-support pyulog
!wget -q https://stmoon.github.io/python-lecture-slides/part2-4-data/flight.mcap -O flight.mcap
!wget -q https://stmoon.github.io/python-lecture-slides/part2-4-data/flight.ulg -O flight.ulg
!wget -q https://stmoon.github.io/python-lecture-slides/part2-4-data/loaders.py -O loaders.py

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from loaders import load_mcap, load_ulog, describe

matplotlib.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 120)

import os
os.makedirs("out", exist_ok=True)
print("준비 완료:", os.path.getsize("flight.mcap"), "bytes /", os.path.getsize("flight.ulg"), "bytes")

## 0. 배경 (10분)

### ROS2 와 mcap

**ROS2** 는 로봇 소프트웨어를 여러 **노드**로 나누고, 노드끼리 **토픽**으로 메시지를 주고받게 하는 미들웨어다.
카메라 노드가 `/image` 토픽으로 사진을 흘리면, 인식 노드가 그것을 받아 쓰는 식이다.

**mcap** 은 그 토픽들을 시간순으로 담아 두는 기록 파일 형식이다 (ROS2 의 `rosbag2` 가 쓰는 기본 형식).
파일 하나 안에 여러 토픽이 들어 있고, 토픽마다 메시지 구조(스키마)가 다르다.
이번 파일에는 네 토픽이 있다.

| 토픽 | 메시지 타입 | 주기 | 내용 |
|---|---|---|---|
| `/imu/data` | `sensor_msgs/msg/Imu` | 25 Hz | 가속도, 각속도, 자세 |
| `/fix` | `sensor_msgs/msg/NavSatFix` | 5 Hz | 위도, 경도, 고도 |
| `/cmd_vel` | `geometry_msgs/msg/Twist` | 10 Hz | 명령 속도 |
| `/temperature` | `sensor_msgs/msg/Temperature` | 1 Hz | 보드 온도 |

### PX4 와 ulog

**PX4** 는 드론의 비행제어 소프트웨어다. 자세를 추정하고 모터 명령을 계산하는 일을 한다.
내부 모듈끼리는 **uORB** 라는 메시지 버스로 값을 주고받고, 그 메시지들을 그대로 파일에 남긴 것이 **ulog** 다.

| 메시지 | 주기 | 내용 |
|---|---|---|
| `sensor_combined` | 50 Hz | 가속도 3축, z축 각속도 |
| `vehicle_local_position` | 20 Hz | 위치와 속도 (NED) |
| `vehicle_attitude` | 20 Hz | roll, pitch, yaw |
| `battery_status` | 2 Hz | 전압, 전류, 잔량 |

### 두 로그의 공통점

이름과 포맷은 다르지만 구조는 같다. **"시각 열 하나 + 값 열 여러 개"인 표가 여러 개** 있는 것이다.
열고 나면 둘 다 DataFrame 이고, 그 다음부터는 Pandas 로 똑같이 다룬다.

주의할 차이 세 가지.

1. **시각 기준이 다르다** — mcap 은 1970년 기준 나노초, ulog 은 부팅 후 마이크로초. 그대로 비교하면 안 된다.
2. **좌표계가 다르다** — ROS2 는 ENU(동-북-상), PX4 는 NED(북-동-하). PX4 의 `z` 는 아래가 양수다.
3. **주기가 다르다** — 25 Hz 와 50 Hz 를 나란히 놓으려면 같은 주기로 다시 샘플링해야 한다.

## 과제 1. 로그 열고 구조 파악 (15분)

두 로그에 어떤 표가 들어 있는지 확인하고, 각 IMU 가 초당 몇 번 기록했는지 구한다.

아래 코드 셀에는 로그를 여는 두 줄이 이미 들어 있다. `topics` 와 `msgs` 는 **딕셔너리**다.
키는 토픽 이름(mcap) 또는 메시지 이름(ulog), 값은 DataFrame 이다.
`describe(topics)` 는 `loaders.py` 가 주는 도우미로 표 목록과 열 이름을 찍어 준다.
DataFrame 의 `df.describe()`(값 요약 통계)와는 다른 함수다.

**할 일**
1. 다섯 개 표를 딕셔너리에서 꺼내 아래 이름의 변수에 담는다. 뒤 과제가 이 이름을 그대로 쓴다.

   | 변수 | 딕셔너리 | 키 | 시각 열 | 시각 단위 |
   |---|---|---|---|---|
   | `imu` | `topics` | `/imu/data` | `t_ns` | 나노초 |
   | `fix` | `topics` | `/fix` | `t_ns` | 나노초 |
   | `sc` | `msgs` | `sensor_combined` | `timestamp` | 마이크로초 |
   | `lp` | `msgs` | `vehicle_local_position` | `timestamp` | 마이크로초 |
   | `att` | `msgs` | `vehicle_attitude` | `timestamp` | 마이크로초 |

2. `imu.head()`, `imu.info()`, `imu.describe()` 를 차례로 출력해 열 이름, 자료형, 값 범위를 본다.
3. IMU 두 개의 **표본 주기(Hz)** 를 각각 구해 출력한다.
   표본 간격은 시각 열 차이의 중앙값 `df[시각열].diff().median()` 이고, 주기는 그 간격의 역수다.
   시각 단위가 다르므로 나누는 수도 다르다.

   - ROS2 `imu` : `1e9 / imu["t_ns"].diff().median()`
   - PX4 `sc` : `1e6 / sc["timestamp"].diff().median()`

**확인 항목** — `topics` 4개, `msgs` 4개 표. `imu` 는 7500행, `sc` 는 15000행.
주기는 각각 25 Hz, 50 Hz 근처.

**흔한 실수** — 나노초는 `1e9`, 마이크로초는 `1e6` 으로 나눈다.
간격은 평균 대신 중앙값으로 잡는다. 튀는 간격 하나가 평균을 흔들기 때문이다.

In [ ]:
topics = load_mcap("flight.mcap")
msgs = load_ulog("flight.ulg")

describe(topics)
describe(msgs)

# 표 다섯 개를 변수에 담는다. 첫 줄이 예시다.
imu = topics["/imu/data"]
fix = ...   # IMPLEMENT HERE
sc = ...    # IMPLEMENT HERE
lp = ...    # IMPLEMENT HERE
att = ...   # IMPLEMENT HERE

display(imu.head())
imu.info()
display(imu.describe())

# 표본 주기(Hz) = 1 / 표본 간격(초). 시각 단위에 맞춰 나눈다.
print("ROS2 IMU  :", round(1e9 / imu["t_ns"].diff().median(), 1), "Hz")
print("PX4 accel :", round(..., 1), "Hz")   # IMPLEMENT HERE

## 과제 2. 시간축 정리 (15분)

두 로그의 시각을 **같은 기준의 경과 초**로 바꾼다.

**할 일**
1. mcap 전체에서 가장 이른 `t_ns` 를 `t0_ns` 로 잡는다. `imu`, `fix` 에 `t = (t_ns - t0_ns) / 1e9` 열을 만든다.
2. ulog 전체에서 가장 이른 `timestamp` 를 `t0_us` 로 잡는다. `sc`, `lp`, `att` 에 `t = (timestamp - t0_us) / 1e6` 열을 만든다.
3. `imu` 에 중복된 시각이 있다. 중복을 제거하고 시각 순으로 정렬한다 (인덱스도 다시 매긴다).
4. `fix` 의 시각 간격(`diff`)을 구해 **정상 간격보다 크게 벌어진 구간**을 찾아 출력한다.

**확인 항목** — 중복 제거 후 `imu` 는 7497행. `fix` 에서 15초짜리 구멍 하나가 나온다.

**흔한 실수** — `drop_duplicates` 는 기본이 모든 열 기준이다. 시각 열을 지정해야 한다.
정렬 뒤 `reset_index(drop=True)` 를 빠뜨리면 뒤 과제에서 인덱스가 어긋난다.

In [ ]:
t0_ns = min(df["t_ns"].min() for df in topics.values())
t0_us = min(df["timestamp"].min() for df in msgs.values())

# 경과 초 열 t 를 만든다. mcap 은 나노초, ulog 은 마이크로초다.
for df in (imu, fix):
    df["t"] = (df["t_ns"] - t0_ns) / 1e9
for df in (sc, lp, att):
    df["t"] = ...   # IMPLEMENT HERE

print("중복 제거 전:", len(imu))
imu = ...           # IMPLEMENT HERE: t_ns 중복 제거, t_ns 로 정렬, 인덱스 재부여
print("중복 제거 후:", len(imu))

gap = fix["t"].diff()
step = gap.median()
print("정상 간격:", round(step, 2), "초")
holes = ...         # IMPLEMENT HERE: 간격이 step 의 2배를 넘는 행만, t 열과 gap 열
display(holes)

## 과제 3. 결측과 이상치 (20분)

값이 이상한 자리를 골라낸다. **물리적으로 불가능한 값을 먼저 지우고**, 그 다음에 통계로 본다.

**할 일**
1. `fix["altitude"]` 에 센티널 `-999.0` 이 섞여 있다. 결측(`NaN`)으로 바꾸고 결측 비율을 출력한다.
2. 바뀐 결측을 앞뒤 값으로 보간한다 (`interpolate`).
3. `sc["accelerometer_m_s2_z"]` 에서 **크기가 40 m/s^2 를 넘는 값**을 결측으로 바꿔 `az` 열을 만든다.
   중력이 9.8 이므로 40 을 넘는 값은 센서 오류다.
4. 만든 결측을 보간한다.
5. 보간 후 `az` 의 최댓값을 확인한다. 20 을 넘는 값 두 개가 남는데, 이것은 **오류가 아니라 실제 충격**이다.
   몇 초에 있었는지 출력한다.

**확인 항목** — altitude 결측 3개. `az` 에서 지워지는 값 12개. 남는 충격 2회.

**흔한 실수** — z-score 로 먼저 자르면 실제 충격까지 지워진다. 물리 범위가 먼저다.
`replace(-999, np.nan)` 은 새 Series 를 돌려준다. 열에 다시 대입해야 반영된다.

In [ ]:
# 1) altitude 의 센티널 -999 를 결측으로 바꾸고 비율을 본다
fix["altitude"] = fix["altitude"].replace(-999.0, np.nan)
print("altitude 결측:", int(fix["altitude"].isna().sum()),
      f"({fix['altitude'].isna().mean() * 100:.2f}%)")
fix["altitude"] = ...   # IMPLEMENT HERE: 앞뒤 값으로 보간

# 2) 가속도 z 에서 크기가 40 을 넘는 값을 결측으로 바꿔 az 열을 만든다
raw = sc["accelerometer_m_s2_z"]
sc["az"] = ...          # IMPLEMENT HERE
print("물리 범위 밖:", int(sc["az"].isna().sum()), "개")
sc["az"] = sc["az"].interpolate()

# 3) 보간 뒤에도 20 을 넘는 값은 오류가 아니라 실제 충격이다
shock = ...             # IMPLEMENT HERE: az > 20 인 행의 t, az 두 열
print("남은 충격:", len(shock), "회")
display(shock.round(2))

## 과제 4. 단위와 좌표 (15분)

두 로그의 좌표계를 맞추고, 센서 장착 오차를 보정한다.

**할 일**
1. `lp` 는 PX4 의 **NED** 좌표다. `x` 가 북, `y` 가 동, `z` 는 **아래가 양수**다.
   ROS2 와 같은 **ENU** 로 바꿔 `east`, `north`, `up` 열을 만든다.
2. `up` 의 최댓값으로 최고 고도를 확인한다.
3. IMU 가 기체 앞방향에서 **15도 틀어져** 장착되어 있다. 2차원 회전 행렬을 만들어
   `imu` 의 `linear_acceleration_x`, `linear_acceleration_y` 를 기체 좌표로 돌려
   `ax_body`, `ay_body` 열에 넣는다.
4. 보정 전후의 x축 가속도 평균을 비교한다.

**회전 행렬** — 각도 `th` 에 대해

```
R = [[cos(th), -sin(th)],
     [sin(th),  cos(th)]]
```

점을 행으로 쌓은 배열 `P` 에는 `P @ R.T` 로 적용한다 (NumPy 연습문제 9 와 같은 방식).

**확인 항목** — 최고 고도 30 m 근처. 보정 후 값이 달라진다.

**흔한 실수** — `up = -z` 다. 부호를 빠뜨리면 고도가 음수로 나온다.
`np.deg2rad` 없이 15 를 그대로 넣으면 라디안이 아니라 15 라디안이 된다.

In [ ]:
# NED(북-동-하) -> ENU(동-북-상)
lp["east"] = lp["y"]
lp["north"] = ...   # IMPLEMENT HERE
lp["up"] = ...      # IMPLEMENT HERE: 부호 주의
print("최고 고도:", round(lp["up"].max(), 1), "m")

# 장착 오차 15도를 되돌린다
th = np.deg2rad(15.0)
R = ...             # IMPLEMENT HERE: 2x2 회전 행렬
xy = imu[["linear_acceleration_x", "linear_acceleration_y"]].to_numpy()
imu[["ax_body", "ay_body"]] = ...   # IMPLEMENT HERE: xy 에 R 적용

print("보정 전 x 평균:", round(imu["linear_acceleration_x"].mean(), 4))
print("보정 후 x 평균:", round(imu["ax_body"].mean(), 4))

## 과제 5. 두 로그 합치기 (20분)

주기가 다른 두 로그를 **같은 시간 격자**에 올려 나란히 본다.

**할 일**
1. `imu` 와 `sc` 의 `t`(경과 초)를 `pd.to_datetime(..., unit="s")` 로 바꿔 인덱스로 세운다.
2. 각각 **1초 평균**으로 리샘플링한다. mcap 쪽은 `linear_acceleration_z`, ulog 쪽은 `az` 를 쓴다.
3. 두 결과를 시간 인덱스 기준으로 결합해 `merged` 를 만든다. 열 이름은 `ros_az`, `px4_az`.
4. 차이 열 `diff = ros_az - px4_az` 를 만들고 `describe()` 로 요약한다.
5. `lp` 의 `up` 도 같은 방식으로 1초 평균을 내어 `merged` 에 `up` 열로 붙인다.
6. `merged` 를 `out/clean.csv` 로 저장한다.

**확인 항목** — `merged` 는 300행 근처. 두 가속도의 차이는 평균 0 근처에서 작게 흔들린다.

**흔한 실수** — 리샘플링은 **시간 인덱스**가 있어야 동작한다. `t` 열만 있고 인덱스가 정수면 오류가 난다.
`merge` 는 인덱스로 붙일 때 `left_index=True, right_index=True` 가 필요하다.

In [ ]:
# 경과 초 t 를 시간 인덱스로 세운 Series 세 개
ros = imu.set_index(pd.to_datetime(imu["t"], unit="s"))["linear_acceleration_z"]
px4 = ...   # IMPLEMENT HERE: sc 의 az
alt = ...   # IMPLEMENT HERE: lp 의 up

merged = pd.merge(ros.resample("1s").mean().rename("ros_az"),
                  ...,   # IMPLEMENT HERE: px4 의 1초 평균, 이름은 px4_az
                  left_index=True, right_index=True)
merged["diff"] = ...     # IMPLEMENT HERE
merged["up"] = alt.resample("1s").mean()

print(merged.shape)
display(merged.head())
display(merged[["ros_az", "px4_az", "diff"]].describe().round(3))

merged.to_csv("out/clean.csv")

## 과제 6. 집계와 시각화 (20분)

비행 구간을 나누고, 구간별 통계와 그림을 만든다.

**할 일**
1. `merged` 에서 `up > 1.0` 이면 비행 중으로 본다. 불리언 열 `flying` 을 만든다.
2. 상태가 바뀔 때마다 번호가 올라가는 구간 라벨을 만든다.
   `(flying != flying.shift()).cumsum()` 이 그 번호다. `seg` 열에 넣는다.
3. `seg` 로 묶어 구간별 **시작 시각(초), 길이(초), 평균 고도, 최대 가속도, 비행 여부**를 담은 표를 만든다.
   `out/segments.csv` 로 저장한다.
4. 2행 2열 대시보드를 그려 `out/dashboard.png` 로 저장한다.
   - 좌상: 시간에 따른 고도 `up`
   - 우상: `ros_az` 와 `px4_az` 두 선 (범례 포함)
   - 좌하: `diff` 히스토그램 (bins=30)
   - 우하: `ros_az` 와 `px4_az` 의 산점도

**확인 항목** — 구간은 3개 (지상 - 비행 - 지상). 비행 구간 길이는 250초 근처.

**흔한 실수** — `groupby` 결과의 열 이름은 `agg(이름="함수")` 로 직접 정하는 편이 읽기 좋다.
`savefig` 는 `show` 보다 **먼저** 불러야 한다. 그린 다음 비워지기 때문이다.

In [ ]:
merged["flying"] = ...   # IMPLEMENT HERE: up 이 1.0 보다 크면 비행 중
merged["seg"] = ...      # IMPLEMENT HERE: 상태가 바뀔 때마다 1씩 오르는 번호

flat = merged.reset_index(names="stamp")
flat["sec"] = (flat["stamp"] - flat["stamp"].min()).dt.total_seconds()
seg = flat.groupby("seg").agg(
    start_s=("sec", "min"),
    seconds=("sec", "count"),
    mean_up=...,    # IMPLEMENT HERE: up 의 평균
    max_az=...,     # IMPLEMENT HERE: ros_az 의 최대
    flying=("flying", "first"),
).round({"start_s": 1, "mean_up": 2, "max_az": 2})
display(seg)
seg.to_csv("out/segments.csv")

fig, ax = plt.subplots(2, 2, figsize=(11, 6))
ax[0, 0].plot(merged.index, merged["up"])
ax[0, 0].set_title("altitude"); ax[0, 0].set_ylabel("up (m)")

# IMPLEMENT HERE: 우상 ax[0, 1] 에 ros_az, px4_az 두 선과 범례
# IMPLEMENT HERE: 좌하 ax[1, 0] 에 diff 히스토그램 (bins=30)
# IMPLEMENT HERE: 우하 ax[1, 1] 에 ros_az 대 px4_az 산점도

fig.tight_layout()
fig.savefig("out/dashboard.png", dpi=120, bbox_inches="tight")
print("saved:", os.path.getsize("out/dashboard.png"), "bytes")

## 과제 7. 패키지로 묶기 (15분)

여기까지 쓴 처리를 **다음 로그에도 그대로 쓸 수 있게** 패키지로 만든다. Part 1 에서 배운 구조다.

**할 일** — 코드 셀에 뼈대가 있다. 파일 두 개의 내용을 문자열로 채우면 아래 줄들이 그것을 파일로 쓴다.

1. `clean.py` 의 `drop_out_of_range(s, lo, hi)` 본문을 채운다.
   범위 `[lo, hi]` 밖 값을 결측으로 바꾸고 보간해 돌려준다. `to_seconds` 는 예시로 이미 채워져 있다.
2. `init_py` 문자열을 채운다. `clean` 의 두 함수를 재노출하고 `__all__` 과 `__version__` 을 정한다.
3. 아래 `import flightkit` 줄이 그대로 돌아가면 된다.
   `flightkit.drop_out_of_range` 로 과제 3 을 다시 돌려 최댓값이 같은지 확인한다.

**확인 항목** — `flightkit.__all__` 이 두 이름을 담는다. 다시 돌린 결과의 최댓값이 과제 3 과 같다.

**흔한 실수** — Colab 에서 파일을 고친 뒤 다시 `import` 해도 예전 것이 그대로 쓰인다 (모듈 캐시).
`importlib.reload` 를 쓰거나 셀을 처음부터 다시 실행한다.

In [ ]:
os.makedirs("flightkit", exist_ok=True)

clean_py = '''
def to_seconds(df, col, t0, scale):
    "Add an elapsed-seconds column t, built from a raw clock column."
    out = df.copy()
    out["t"] = (out[col] - t0) / scale
    return out


def drop_out_of_range(s, lo, hi):
    "Blank out values outside [lo, hi], then fill the holes by interpolation."
    return None   # IMPLEMENT HERE
'''

# IMPLEMENT HERE: 두 함수를 재노출하고 __all__ 과 __version__ 을 정한다
init_py = '''
'''

open("flightkit/clean.py", "w").write(clean_py)
open("flightkit/__init__.py", "w").write(init_py)

import flightkit
print(flightkit.__all__, flightkit.__version__)

again = flightkit.drop_out_of_range(sc["accelerometer_m_s2_z"], -40, 40)
print("과제 3 결과:", round(sc["az"].max(), 3), "/ 패키지 결과:", round(again.max(), 3))